# Squat Video Annotation

Label segmented reps using local data from the CLI pipeline:

```bash
python run_annotation.py extract
python run_annotation.py segment
python -m jupyter lab notebooks/annotate.ipynb
python run_annotation.py export --check
```

Data lives under `data/` (videos, landmarks, rep cache, annotations).


## 1. Setup

In [1]:
# Dependencies: use the project venv (pip install -r requirements.txt)
# %pip install -q ipywidgets opencv-python-headless


In [2]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "annotation").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output

import annotation.config as _annotation_config
importlib.reload(_annotation_config)
from annotation.config import AnnotationConfig
from annotation.cache import (
    list_videos,
    load_landmark_cache,
    load_reps_cache,
    save_reps_cache,
    landmark_cache_path,
    resolve_video_path,
)
from annotation.rep_segmenter import RepSegmenter
from annotation.store import AnnotationStore
from annotation.exporter import DataExporter
from annotation.video_display import orient_frame_for_display
import annotation.interactive as _annotation_interactive
importlib.reload(_annotation_interactive)

print(f"Project root: {PROJECT_ROOT}")
print(f"annotation package: {(PROJECT_ROOT / 'annotation').is_dir()}")





Project root: E:\Repos\form-fit-ai\squat-posture-ai
annotation package: True


In [3]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab: set cfg.base_dir to your Drive folder in the next cell.")
else:
    print("Local mode — data/ under project root.")


Local mode — data/ under project root.


In [4]:
cfg = AnnotationConfig(base_dir=PROJECT_ROOT / "data")
cfg.ensure_dirs()

segmenter = RepSegmenter(cfg)

print(f"Videos:     {cfg.video_dir.resolve()}")
print(f"Landmarks:  {cfg.landmarks_dir.resolve()}")
print(f"Annotated:  {cfg.output_dir.resolve()}")
print(f"Rep cache:  {cfg.reps_cache_path.resolve()}")


Videos:     E:\Repos\form-fit-ai\squat-posture-ai\data\raw\videos
Landmarks:  E:\Repos\form-fit-ai\squat-posture-ai\data\landmarks
Annotated:  E:\Repos\form-fit-ai\squat-posture-ai\data\annotated
Rep cache:  E:\Repos\form-fit-ai\squat-posture-ai\data\landmarks\all_reps_cache.json


## 2. Load videos & landmark caches

Videos: `data/raw/videos/`. Landmarks: `data/landmarks/` (from `run_annotation.py extract`).


In [5]:
video_files = list_videos(cfg.video_dir)
print(f"Found {len(video_files)} video(s) in {cfg.video_dir.resolve()}")
for vf in video_files[:10]:
    print(f"  {vf.name}")
if len(video_files) > 10:
    print(f"  ... and {len(video_files) - 10} more")


Found 167 video(s) in E:\Repos\form-fit-ai\squat-posture-ai\data\raw\videos
  VID_20260330_110816.mp4
  VID_20260330_110839.mp4
  VID_20260330_110851.mp4
  VID_20260330_110904.mp4
  VID_20260330_110942.mp4
  VID_20260330_111000.mp4
  VID_20260330_111016.mp4
  VID_20260330_111030.mp4
  VID_20260330_111054.mp4
  VID_20260330_111105.mp4
  ... and 157 more


## 3. Landmark caches

Run `python run_annotation.py extract` if caches are missing. This cell loads existing `.npz` files only.


In [6]:
# Pose extraction is handled by the CLI (run_annotation.py extract).
# Skipped here — we load cached landmarks below.


## 4. Load cached landmarks


In [7]:
# Optional Colab remount — ignored locally
# from google.colab import drive; drive.mount('/content/drive', force_remount=True)


In [8]:
extracted_data = {}

for video_path in video_files:
    video_name = video_path.stem
    cache_path = landmark_cache_path(cfg.landmarks_dir, video_name)
    cached = load_landmark_cache(cache_path)
    if cached is None:
        print(f"  missing cache: {video_name} (run extract)")
        continue
    extracted_data[video_name] = cached

print(f"\nLoaded {len(extracted_data)} landmark cache(s)")
for name, data in list(extracted_data.items())[:5]:
    print(f"  {name}: {data['landmarks'].shape[0]} frames")
if len(extracted_data) > 5:
    print(f"  ... and {len(extracted_data) - 5} more")



Loaded 167 landmark cache(s)
  VID_20260330_110816: 456 frames
  VID_20260330_110839: 239 frames
  VID_20260330_110851: 206 frames
  VID_20260330_110904: 206 frames
  VID_20260330_110942: 215 frames
  ... and 162 more


## 5. Rep segmentation

Uses `RepSegmenter` from `annotation/` (same logic as mobile). Cache: `data/landmarks/all_reps_cache.json`.


In [9]:
# RepSegmenter imported above — knee-based FSM aligned with mobile app.


## 6. Load or run segmentation


In [10]:
FORCE_RESEGMENT = False  # True clears cache and re-segments (invalidates label indices)

if FORCE_RESEGMENT and cfg.reps_cache_path.exists():
    cfg.reps_cache_path.unlink()
    print("Rep cache deleted.")

all_reps = load_reps_cache(cfg.reps_cache_path)

if all_reps is None:
    print("No rep cache — segmenting...")
    all_reps = {}
    for video_name, data in extracted_data.items():
        reps = segmenter.segment_reps(data["landmarks"])
        all_reps[video_name] = reps
        print(f"  {video_name}: {len(reps)} rep(s)")
    save_reps_cache(all_reps, cfg.reps_cache_path)
else:
    n = sum(len(v) for v in all_reps.values())
    print(f"Loaded rep cache — {len(all_reps)} video(s), {n} rep(s)")

print(f"\nTotal reps: {sum(len(r) for r in all_reps.values())}")


Loaded rep cache — 167 video(s), 481 rep(s)

Total reps: 481


## 7. Manual Rep Boundary Adjustment

In [11]:
def manually_add_rep(video_name, start_frame, end_frame):
    """Manually add a rep that the segmenter missed."""
    if video_name not in all_reps:
        all_reps[video_name] = []
    all_reps[video_name].append((start_frame, end_frame))
    all_reps[video_name].sort(key=lambda x: x[0])
    print(f"Added rep [{start_frame}:{end_frame}] to {video_name}")


def manually_remove_rep(video_name, rep_index):
    """Remove a false-positive rep by index (0-based)."""
    if video_name in all_reps and 0 <= rep_index < len(all_reps[video_name]):
        removed = all_reps[video_name].pop(rep_index)
        print(f"Removed rep {rep_index} from {video_name}")


def adjust_rep_boundary(video_name, rep_index, new_start=None, new_end=None):
    """Fine-tune rep start/end boundaries."""
    if video_name in all_reps and 0 <= rep_index < len(all_reps[video_name]):
        start, end = all_reps[video_name][rep_index]
        if new_start is not None:
            start = new_start
        if new_end is not None:
            end = new_end
        all_reps[video_name][rep_index] = (start, end)
        print(f"Adjusted rep {rep_index} in {video_name} to [{start}:{end}]")


# Example usage:
# manually_add_rep('my_video', start_frame=120, end_frame=195)
# manually_remove_rep('my_video', rep_index=2)
# adjust_rep_boundary('my_video', rep_index=0, new_start=10, new_end=85)

print("Manual adjustment functions ready.")

Manual adjustment functions ready.


## 8. Annotation store

Persists labels to `data/annotated/annotations.json`.


In [12]:
store = AnnotationStore(cfg.output_dir)
progress = store.get_progress(all_reps)
print(f"Progress: {progress['annotated']}/{progress['total_reps']} ({progress['progress_pct']:.0f}%)")


Progress: 411/481 (85%)


## 9. Visual Annotation Tool

In [13]:
import importlib
import annotation.interactive as _interactive
importlib.reload(_interactive)
from annotation.interactive import build_pending_video_list, run_annotation_session

rotation = getattr(cfg, "video_display_rotation_cw", 90)
video_list = build_pending_video_list(cfg.video_dir, extracted_data, all_reps, store)

if not video_list:
    print("All videos fully annotated. Nothing left to do.")
else:
    print(f"Resuming: {len(video_list)} video(s) still need annotation")
    run_annotation_session(store, video_list, rotation_cw=rotation)



  done: VID_20260330_110816 (5 reps)
  done: VID_20260330_110839 (4 reps)
  done: VID_20260330_110851 (4 reps)
  done: VID_20260330_110904 (3 reps)
  done: VID_20260330_110942 (4 reps)
  done: VID_20260330_111000 (4 reps)
  done: VID_20260330_111016 (6 reps)
  done: VID_20260330_111030 (4 reps)
  done: VID_20260330_111054 (3 reps)
  done: VID_20260330_111105 (4 reps)
  done: VID_20260330_111116 (4 reps)
  done: VID_20260330_111127 (5 reps)
  done: VID_20260330_111200 (4 reps)
  done: VID_20260330_111213 (2 reps)
  done: VID_20260330_111225 (3 reps)
  done: VID_20260330_111238 (3 reps)
  done: VID_20260330_111303 (4 reps)
  done: VID_20260330_111315 (4 reps)
  done: VID_20260330_111326 (3 reps)
  done: VID_20260330_111339 (3 reps)
  done: VID_20260330_111353 (1 reps)
  done: VID_20260330_111406 (5 reps)
  done: VID_20260330_111418 (4 reps)
  done: VID_20260330_111429 (5 reps)
  done: VID_20260330_111441 (4 reps)
  done: VID_20260330_111500 (1 reps)
  done: VID_20260330_111511 (4 reps)
 

Output()

## 11. Batch Annotation (Programmatic)

In [14]:
def batch_annotate(video_name, labels):
    """Annotate multiple reps from a list of label dicts.

    Example:
        batch_annotate('squat_video_1', [
            {'correct': True},
            {'correct': False, 'knee_valgus': True},
            {'correct': False, 'insufficient_depth': True, 'forward_lean': True},
        ])
    """
    if video_name not in all_reps:
        print(f"No reps for '{video_name}'. Run segmentation first.")
        return

    reps = all_reps[video_name]
    num_to_annotate = min(len(labels), len(reps))

    for idx in range(num_to_annotate):
        start, end = reps[idx]
        label = labels[idx]
        store.annotate_rep(
            video_name=video_name,
            rep_index=idx,
            start_frame=start,
            end_frame=end,
            is_correct=label.get('correct', True),
            knee_valgus=label.get('knee_valgus', False),
            insufficient_depth=label.get('insufficient_depth', False),
            forward_lean=label.get('forward_lean', False),
            notes=label.get('notes', ''),
        )

    store.save()
    print(f"Batch annotated {num_to_annotate} reps for '{video_name}'.")


# Example:
# batch_annotate('my_squat_video', [
#     {'correct': True},
#     {'correct': False, 'knee_valgus': True},
# ])

print("batch_annotate() ready.")

batch_annotate() ready.


## 12. Export for training

Writes `data/annotated/annotated_dataset.npz`.


In [15]:
# DataExporter imported from annotation.exporter


In [16]:
exporter = DataExporter(cfg)
output_path = exporter.export_for_training(extracted_data, store.annotations)

if output_path:
    print(f"Exported: {output_path}")
    print("In train.ipynb:")
    print(f"  data = np.load(r'{output_path}')")


Exported: E:\Repos\form-fit-ai\squat-posture-ai\data\annotated\annotated_dataset.npz
In train.ipynb:
  data = np.load(r'E:\Repos\form-fit-ai\squat-posture-ai\data\annotated\annotated_dataset.npz')


## 13. Loading in Training Notebook

Replace the synthetic data generation cell in `squat_posture_analysis.ipynb` with:

```python
data = np.load('./annotated_data/annotated_dataset.npz', allow_pickle=True)
raw_sequences = data['sequences']     # (N, 45, 132)
labels = data['labels']               # (N,)
error_vectors = data['error_vectors'] # (N, 3)

print(f"Loaded: {raw_sequences.shape[0]} annotated sequences")
```

Everything downstream works unchanged.

---

## Workflow Summary

1. Place videos in `./videos/`
2. Run cells 1-6 to extract landmarks and segment reps
3. Adjust boundaries (Section 7) if needed
4. Annotate: interactive (Section 9-10) or batch (Section 11)
5. Export (Section 12)
6. Load in training notebook

### Output Files

```
annotated_data/
  annotations.json          # Human-readable labels
  annotated_dataset.npz     # Training-ready arrays

extracted_landmarks/
  {video}_landmarks.npz     # Cached pose data
```